## Прогон speed‑классификатора по ROI из `roi_index_test.jsonl`

Этот ноутбук:
- читает `speed_test_roi/roi_index_test.jsonl`
- для каждого объекта открывает файл `roi_path`
- прогоняет через PyTorch‑модель speed‑классификатора
- сохраняет новый файл с результатами: `roi_path, true_class_name, class_name, confidence`

### Важно
- `true_class_name` — **истинный** класс из `roi_index_test.jsonl`.
- `class_name` в выходном файле — **предсказанный** класс (например `3.24_50`).
- Это честная оценка только если в обучение подмешивается `roi_index_train.jsonl`.
- Препроцессинг совпадает с обучением: resize → `ToTensor()` → `Normalize(mean,std)`.


In [ ]:
# Параметры: пути и настройки

from pathlib import Path

# Папка с ROI и индексом
ROI_DIR = Path("speed_test_roi")
INDEX_JSONL = ROI_DIR / "roi_index_test.jsonl"

# Эталонный список классов (в том порядке, на котором обучалась модель)
LABELS_TXT = Path("datasets/speed_cls_v1/labels.txt")

# Какой чекпоинт использовать (PyTorch)
CKPT_PATH = Path("models/speed_classifier/run_1/best.pt")

# Размер входа модели
IMAGE_SIZE = 128

# Куда сохранить итоговый файл
OUT_CSV = ROI_DIR / "roi_predictions_test.csv"

# Сколько примеров прогнать (None = все)
LIMIT = None


In [ ]:
# Импорты + подготовка окружения

import csv
import json
import os
import sys

import numpy as np
import torch
from PIL import Image

# Чтобы импорты training.* работали из любого cwd
REPO_ROOT = Path(os.getcwd()).resolve()
while not (REPO_ROOT / "training").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from py_utils.device import pick_device
from training.speed_classifier.model import create_model


def build_transform(image_size: int):
    """Препроцессинг как на обучении: Resize -> ToTensor(0..1) -> Normalize(ImageNet)."""

    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]

    def _tfm(pil_img: Image.Image) -> torch.Tensor:
        img = pil_img.convert("RGB").resize((int(image_size), int(image_size)))
        x = torch.from_numpy(np.array(img, dtype=np.uint8)).permute(2, 0, 1).contiguous()
        x = x.to(torch.float32) / 255.0
        x = (x - mean) / std
        return x

    return _tfm


@torch.no_grad()
def predict_one(
    *,
    model: torch.nn.Module,
    x_chw: torch.Tensor,
    device: torch.device,
    classes: list[str],
) -> tuple[str, float]:
    """Возвращает (predicted_class_name, confidence_softmax)."""

    logits = model(x_chw.unsqueeze(0).to(device))
    probs = torch.softmax(logits, dim=1)[0]
    idx = int(torch.argmax(probs).item())
    conf = float(probs[idx].item())
    name = classes[idx] if 0 <= idx < len(classes) else str(idx)
    return name, conf


In [ ]:
# Загрузка модели

if not INDEX_JSONL.is_file():
    raise FileNotFoundError(f"Не найден индекс: {INDEX_JSONL}")
if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Не найден чекпоинт: {CKPT_PATH}")

ckpt = torch.load(str(CKPT_PATH), map_location="cpu")
classes = ckpt.get("classes")
print (classes)
if not isinstance(classes, list) or len(classes) < 2:
    raise RuntimeError("В чекпоинте нет списка classes (ckpt['classes']).")
classes = [str(x) for x in classes]

device = torch.device(pick_device())
print("device:", device)
print("num_classes:", len(classes))

model = create_model(len(classes), pretrained=False).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

tfm = build_transform(IMAGE_SIZE)


In [ ]:
# Проверки: соответствие индексов классов обучению + согласованность с ROI-индексом

from typing import Iterable


def read_labels_txt(path: Path) -> list[str]:
    if not path.is_file():
        raise FileNotFoundError(f"Не найден файл labels: {path}")
    labels: list[str] = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        s = raw.strip()
        if not s or s.startswith("#"):
            continue
        labels.append(s)
    return labels


# 1) Базовые sanity-checks списка classes из чекпоинта
assert isinstance(classes, list) and len(classes) > 1
assert all(isinstance(x, str) and x for x in classes)
assert len(classes) == len(set(classes)), "В ckpt['classes'] есть дубликаты — это почти всегда ошибка"
print("num_classes (ckpt):", len(classes))
print("first ckpt classes:", classes[:10])

# 2) Проверка: истинные классы (true_class_name) должны входить в classes
true_set: set[str] = set()
with INDEX_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        t = obj.get("true_class_name")
        if t:
            true_set.add(str(t))

missing_true = sorted(true_set - set(classes))
print("unique true_class_name:", len(true_set))
if missing_true:
    print("WARNING: true_class_name отсутствуют в ckpt['classes'] (первые 30):")
    print(missing_true[:30])
else:
    print("[OK] все true_class_name присутствуют в ckpt['classes']")

# 3) Проверка соответствия labels.txt (на чём обучали)
# Если чекпоинт обучен на подмножестве классов (например, ROI-адаптация), размеры могут отличаться.
trained_labels = read_labels_txt(LABELS_TXT)
print("num_classes (labels.txt):", len(trained_labels))
print("first labels.txt:", trained_labels[:10])

if len(classes) != len(trained_labels):
    print(
        "[INFO] ckpt['classes'] != labels.txt по размеру. "
        "Похоже, чекпоинт обучался на подмножестве классов — строгую проверку порядка пропускаю."
    )
else:
    if classes == trained_labels:
        print("[OK] ckpt['classes'] полностью совпадает с labels.txt (размер и порядок)")
    else:
        print("ERROR: ckpt['classes'] НЕ совпадает с labels.txt")

        # Печатаем первые расхождения по позициям
        limit = min(len(classes), len(trained_labels))
        shown = 0
        for idx in range(limit):
            if classes[idx] != trained_labels[idx]:
                print(f"  - first diffs @idx={idx}: ckpt={classes[idx]!r} labels={trained_labels[idx]!r}")
                shown += 1
                if shown >= 10:
                    break

        missing_in_ckpt = sorted(set(trained_labels) - set(classes))
        extra_in_ckpt = sorted(set(classes) - set(trained_labels))
        if missing_in_ckpt:
            print("  - missing in ckpt (first 30):", missing_in_ckpt[:30])
        if extra_in_ckpt:
            print("  - extra in ckpt (first 30):", extra_in_ckpt[:30])


In [ ]:
# Прогон по jsonl и сохранение результата

rows: list[tuple[str, str, str, float]] = []

with INDEX_JSONL.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if LIMIT is not None and i >= int(LIMIT):
            break
        line = line.strip()
        if not line:
            continue

        obj = json.loads(line)
        roi_rel = obj.get("roi_path")
        if not roi_rel:
            continue

        true_class_name = obj.get("true_class_name")
        true_class_name = str(true_class_name) if true_class_name else ""

        roi_path = (ROI_DIR / str(roi_rel)).resolve()
        if not roi_path.is_file():
            raise FileNotFoundError(f"ROI не найден: {roi_path}")

        img = Image.open(roi_path)
        x = tfm(img)  # CHW float32

        pred_name, conf = predict_one(model=model, x_chw=x, device=device, classes=classes)
        rows.append((str(roi_path), true_class_name, pred_name, float(conf)))

# Пишем CSV
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUT_CSV.open("w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["roi_path", "true_class_name", "class_name", "confidence"])
    w.writerows(rows)

print("[OK] wrote", OUT_CSV)
print("rows:", len(rows))

# Быстрый просмотр первых строк
rows[:5]
